# Customer Payment Details EDA

This notebook performs an exploratory analysis of `custpaydetails.csv.csv`, which was exported from `custpaydetails.sql`. The analysis focuses on field-by-field representative data, ranges, missingness patterns, date semantics, and basic cross-field consistency checks.

## Executive Summary

The extract contains 194,309 rows and 53 fields.

It is a UDOT-only result set: CUSTOMER has 1 unique value(s), matching the active CTE in the SQL file.

The SQL requires non-null work posting and pay estimate statuses, so posting/pay-app fields are expected to be dense while left-joined descriptive fields can be sparse.

Exact duplicate rows: 0 across 0 duplicate groups.

Always-missing fields: PHASEITEMSTARTDATE, PHASEITEMENDDATE, CONTRACTTYPE, CONTRACTITEMPHASENAME, LINKEDBUDGETITEMID, LINKEDBUDGETITEM, LINKEDBUDGETITEMMODULE, LINKEDBUDGETITEMDESCRIPTION, LINKEDBUDGETITEMCONTAINERPATH, LINKEDBUDGETITEMCONTAINERNAME, LINKEDBUDGETITEMPHASENAME.

Fields with no missing values: 35 of 53.

Primary interpretation: each row is a UDOT contract pay item joined to a work posting, pay estimate, and pay estimate detail row. The result is not one row per project, contract, item, or pay application; the grain is closer to a paid posting/detail line for a contract item.

## SQL-Derived Context

The active query defines a `udot_contract_payment_details` CTE and selects from it. Other customer blocks are present but commented out. The CTE starts with `CORITEMItemDetails` contract items, joins contracts and projects as required tables, then left-joins lookup tables, recursive container paths, linked budget items, commitments, work postings, pay estimate details, and pay estimates. The `WHERE` clause requires `WP.STATUS IS NOT NULL`, `PE.STATUS IS NOT NULL`, and `ContractEndDate < CURRENT_DATE`.

| UDOT source table referenced |
| --- |
| CONTMGTMaster |
| CORITEMContainer |
| CORITEMItemDetails |
| LibraryProjectType |
| LibraryTypeOfContract |
| PROCMGTCommitments |
| PROCMGTPEDetails |
| PROCMGTPayEstimates |
| PROCMGTWorkPosting |
| PROJECTMEASUREMENTUNITS |
| PROJECTPROJECTMAIN |
| PROJECTProjectPhase |
| PROJECTProjectPhaseItem |
| PROJECTProjectStatus |
| projectmeasurementsystems |

In [ ]:
import csv
from collections import Counter, defaultdict
from datetime import datetime
from decimal import Decimal, InvalidOperation
from pathlib import Path

CSV_PATH = Path('custpaydetails.csv.csv')
with CSV_PATH.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
    fields = reader.fieldnames

print(f'Rows: {len(rows):,}')
print(f'Fields: {len(fields):,}')
print(fields)


## Field Inventory

| Field | Inferred role | Missing | Missing % | Unique | Representative examples | Most common values | SQL/source meaning | Missingness interpretation |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| CUSTOMER | categorical/text | 0 | 0.00% | 1 | UDOT | UDOT (194,309) | Literal from query. Active export is the UDOT CTE only. | Never missing in this extract. |
| PROJECTNAME | categorical/text | 0 | 0.00% | 1,341 | SR-9; Rockville to Springdale, Overlay & Shoulders; US-89; MP 423.30 - 428.77; I-15; MP 365.51 - 373.26 | I-15; SR-232 to I-84 (4,014), SR-30; SR-23 to SR-252 (3,497), SR-108; 300 North to 1800 North (2,207) | PROJECTPROJECTMAIN.PROJECTNAME. | Never missing in this extract. |
| PROJECTCODE | categorical/text | 0 | 0.00% | 1,346 | F-0009(41)27; F-0089(419)423; F-I15-8(156)366 | F-I15-7(328)332 (4,014), F-0030(69)102 (3,497), S-0108(36)6 (2,207) | PROJECTPROJECTMAIN.PROJECTCODE. | Never missing in this extract. |
| PROJECTDESC | categorical/text | 163 | 0.08% | 154 | SR-9; Rockville to Springdale, Overlay & Shoulders<br>Minor Rehabilitation - Roadway; Adding a lane/shoulder; Minor Rehabilitation - Roadway | Preservation High Volume (26,265), Rehabilitation High Volume (22,016), Widen Existing Facility (17,167) | PROJECTPROJECTMAIN.DESCRIPTION. | Missing in 163 rows (0.08%). Often co-missing with PHASEITEMSTARTDATE (163), PHASEITEMENDDATE (163), CONTRACTTYPE (163). |
| PROJECTTYPE | categorical/text | 90,969 | 46.82% | 7 | CMGC; Design Build; Design Bid Build | Design Bid Build (95,410), CMGC (5,164), Design Build (2,036) | LibraryProjectType.PROJECTTYPE via PM.ProjectTypeID. | Missing in 90,969 rows (46.82%). Highest clear concentration: PROJECTTYPE=<missing> has 90,969/90,969 missing (100.00%). Often co-missing with PHASEITEMSTARTDATE (90,969), PHASEITEMENDDATE (90,969), CONTRACTTYPE (90,969). |
| PROJECTSTATUS | categorical/text | 9 | 0.00% | 3 | Construction; Complete; Notice To Proceed (NTP) | Construction (192,688), Complete (1,598), Notice To Proceed (NTP) (14) | PROJECTProjectStatus.StatusName via PM.StatusId. | Missing in 9 rows (0.00%). Often co-missing with PROJECTDESC (9), PROJECTTYPE (9), CONTRACTDESCRIPTION (9). |
| CONTRACTID | identifier | 0 | 0.00% | 1,346 | 135; 163; 150 | 747 (4,014), 1562 (3,497), 1416 (2,207) | CONTMGTMaster.ID joined from CORITEMItemDetails.PARENTINSTANCEID. | Never missing in this extract. |
| CONTRACTNAME | categorical/text | 0 | 0.00% | 1,340 | SR-9; Rockville to Springdale, Overlay & Shoulders; US-89; MP 423.30 - 428.77; I-15; MP 365.51 - 373.26 | I-15; SR-232 to I-84 (4,014), SR-30; SR-23 to SR-252 (3,497), SR-108; 300 North to 1800 North (2,207) | CONTMGTMaster.NAME. | Never missing in this extract. |
| CONTRACTDESCRIPTION | categorical/text | 99,729 | 51.32% | 108 | SR-9; Rockville to Springdale, Overlay & Shoulders<br>Minor Rehabilitation - Roadway; Minor Rehabilitation - Roadway<br>SR-16; MP 6.00 - 19.81 & SR-16; MP 6.00 - 19.81; Preservation High Volume | Preservation High Volume (17,653), Widen Existing Facility (9,733), Rehabilitation High Volume (8,366) | CONTMGTMaster.DESC. | Missing in 99,729 rows (51.32%). Highest clear concentration: CONTRACTITEMUNIT=pound has 543/543 missing (100.00%). Often co-missing with PHASEITEMSTARTDATE (99,729), PHASEITEMENDDATE (99,729), CONTRACTTYPE (99,729). |
| CONTRACTSTARTDATE | date/time | 0 | 0.00% | 709 | 2017-01-10 07:00:00.000; 1900-01-01 07:00:00.000; 2017-02-01 07:00:00.000 | 1900-01-01 07:00:00.000 (5,020), 2024-07-08 06:00:00.000 (4,854), 2019-05-17 06:00:00.000 (4,014) | CONTMGTMaster.StartDt. | Never missing in this extract. |
| CONTRACTENDDATE | date/time | 0 | 0.00% | 1,073 | 2017-03-30 00:00:00.000; 1900-08-03 00:00:00.000; 2017-03-28 01:00:00.000 | 2020-06-13 00:00:00.000 (4,147), 2025-07-17 00:00:00.000 (3,497), 2025-11-04 00:00:00.000 (2,207) | CONTMGTMaster.CLOSUREDT; query filters ContractEndDate < CURRENT_DATE. | Never missing in this extract. |
| PROJSTARTDATE | date/time | 0 | 0.00% | 341 | 2017-01-10 07:00:00.000; 2017-03-06 07:00:00.000; 2016-12-09 07:00:00.000 | 1900-01-01 07:00:00.000 (114,496), 2023-06-01 06:00:00.000 (3,864), 2017-05-01 06:00:00.000 (2,362) | PROJECTPROJECTMAIN.STARTDATE. | Never missing in this extract. |
| PROJENDDATE | date/time | 0 | 0.00% | 607 | 2017-03-30 06:00:00.000; 2018-12-31 07:00:00.000; 2100-01-01 07:00:00.000 | 2100-01-01 07:00:00.000 (41,196), 1901-01-29 07:00:00.000 (4,014), 2024-06-09 06:00:00.000 (3,497) | PROJECTPROJECTMAIN.ENDDATE. | Never missing in this extract. |
| PHASEITEMSTARTDATE | date/time | 194,309 | 100.00% | 0 |  |  | PROJECTProjectPhaseItem.STARTDATE through contract-item container hierarchy. | Always missing in this extract. |
| PHASEITEMENDDATE | date/time | 194,309 | 100.00% | 0 |  |  | PROJECTProjectPhaseItem.ENDDATE through contract-item container hierarchy. | Always missing in this extract. |
| WPPOSTINGDATE | date/time | 0 | 0.00% | 20,468 | 2017-03-02 07:00:00.000; 2017-03-15 06:00:00.000; 2017-03-17 06:00:00.000 | 2025-06-30 06:00:00.000 (558), 2020-06-30 06:00:00.000 (445), 2021-06-30 06:00:00.000 (419) | PROCMGTWorkPosting.POSTINGDATE, selected separately from POSTINGDATE. | Never missing in this extract. |
| CONTRACTTYPE | categorical/text | 194,309 | 100.00% | 0 |  |  | LibraryTypeOfContract.TypeOfContract. | Always missing in this extract. |
| ITEMID | identifier | 0 | 0.00% | 66,314 | 16946; 16947; 16949 | 80666 (404), 89748 (258), 80665 (209) | CORITEMItemDetails.ItemID for the contract pay item. | Never missing in this extract. |
| STANDARDITEMNO | categorical/text | 0 | 0.00% | 9,081 | 02721002P; 02721003P; 02724001* | 015017010 (3,094), 01554700* (2,273), 028917320 (2,147) | CORITEMItemDetails.StandardItemNo for the contract pay item. | Never missing in this extract. |
| ITEMNAME | categorical/text | 0 | 0.00% | 17,332 | Untreated Base Course (Plan Quantity); Untreated Base Course (Shouldering); Cold Weather Protection | Traffic Control (5,425), HMA - 1/2 inch (4,411), Mobilization (3,759) | CORITEMItemDetails.DESCRIPTION for the contract pay item. | Never missing in this extract. |
| ITEMUNITPRICE | money/rate | 0 | 0.00% | 16,532 | 42.2500; 1.4500; 1.1500 | 1.0000 (2,128), 10.0000 (1,863), 100.0000 (1,111) | CORITEMItemDetails.UnitPrice. | Never missing in this extract. |
| ITEMTOTALQUANTITY | quantity/number | 0 | 0.00% | 9,077 | 2903.0000; 12132.0000; 6687.0000 | 1.0000 (35,933), 2.0000 (6,048), 4.0000 (3,594) | CORITEMItemDetails.ContractQuantity. | Never missing in this extract. |
| CONTRACTITEMCONTAINERPATH | categorical/text | 13 | 0.01% | 639 | 10 - ROADWAY; 10 - ROADWAY - Trail (South Salt Lake); 30 - LANDSCAPING - Landscaping at Interchange (UDOT) | 10 - ROADWAY (127,649), 40 - SIGNING (28,258), 20 - STRUCTURES (4,638) | Recursive CORITEMContainer full path for CONTMGT. | Missing in 13 rows (0.01%). Often co-missing with PHASEITEMSTARTDATE (13), PHASEITEMENDDATE (13), CONTRACTTYPE (13). |
| CONTRACTITEMCONTAINERNAME | categorical/text | 13 | 0.01% | 520 | 10 - ROADWAY; 10 - ROADWAY - Trail (South Salt Lake); 30 - LANDSCAPING - Landscaping at Interchange (UDOT) | 10 - ROADWAY (128,106), 40 - SIGNING (28,286), 20 - STRUCTURES (4,673) | Leaf CORITEMContainer.ContainerName for CONTMGT. | Missing in 13 rows (0.01%). Often co-missing with PHASEITEMSTARTDATE (13), PHASEITEMENDDATE (13), CONTRACTTYPE (13). |
| CONTRACTITEMSYSTEMNAME | categorical/text | 0 | 0.00% | 1 | IS System | IS System (194,309) | projectmeasurementsystems.SYSTEMNAME through contract measurement system. | Never missing in this extract. |
| CONTRACTITEMUNIT | categorical/text | 13 | 0.01% | 37 | cu yd; ft; sq yd | Each (62,796), ft (24,794), Lump (17,553) | PROJECTMEASUREMENTUNITS.Unit. | Missing in 13 rows (0.01%). Often co-missing with PROJECTTYPE (13), CONTRACTDESCRIPTION (13), PHASEITEMSTARTDATE (13). |
| CONTRACTITEMPHASENAME | categorical/text | 194,309 | 100.00% | 0 |  |  | PROJECTProjectPhaseItem.Phase through contract-item container hierarchy. | Always missing in this extract. |
| LINKEDBUDGETITEMID | identifier | 194,309 | 100.00% | 0 |  |  | Linked CORITEMItemDetails.ItemID when CI.LinkedBudgetItem points to BDGTEST/BDGTREV. | Always missing in this extract. |
| LINKEDBUDGETITEM | categorical/text | 194,309 | 100.00% | 0 |  |  | Linked budget item StandardItemNo. | Always missing in this extract. |
| LINKEDBUDGETITEMMODULE | categorical/text | 194,309 | 100.00% | 0 |  |  | Linked budget item ModuleID, expected BDGTEST or BDGTREV when present. | Always missing in this extract. |
| LINKEDBUDGETITEMDESCRIPTION | categorical/text | 194,309 | 100.00% | 0 |  |  | Linked budget item DESCRIPTION. | Always missing in this extract. |
| LINKEDBUDGETITEMCONTAINERPATH | categorical/text | 194,309 | 100.00% | 0 |  |  | Recursive CORITEMContainer full path for linked budget item. | Always missing in this extract. |
| LINKEDBUDGETITEMCONTAINERNAME | categorical/text | 194,309 | 100.00% | 0 |  |  | Leaf budget container name. | Always missing in this extract. |
| LINKEDBUDGETITEMPHASENAME | categorical/text | 194,309 | 100.00% | 0 |  |  | PROJECTProjectPhaseItem.Phase for linked budget item. | Always missing in this extract. |
| POSTINGID | identifier | 0 | 0.00% | 194,271 | 1438; 1446; 1447 | 5055 (2), 5056 (2), 5057 (2) | PROCMGTWorkPosting.WPostingID. | Never missing in this extract. |
| POSTINGQTY | quantity/number | 0 | 0.00% | 46,698 | 2903.0000; 9969.0000; 6282.9000 | 1.0000 (32,596), 2.0000 (9,333), 0.2500 (5,890) | PROCMGTWorkPosting.PostingQty. | Never missing in this extract. |
| POSTINGUNITRATE | money/rate | 0 | 0.00% | 16,532 | 42.2500; 1.4500; 1.1500 | 1.0000 (2,128), 10.0000 (1,863), 100.0000 (1,111) | PROCMGTWorkPosting.UnitRate. | Never missing in this extract. |
| TOTALPOSTINGAMOUNT | money/rate | 0 | 0.00% | 91,207 | 122651.75000000; 14455.05000000; 7225.33500000 | 0.00000000 (3,456), 1500.00000000 (539), 3000.00000000 (523) | Computed in SQL as WP.PostingQty * WP.UnitRate. | Never missing in this extract. |
| REFERENCEPOSTINGTYPE | categorical/text | 0 | 0.00% | 1 | ITMPOST | ITMPOST (194,309) | PROCMGTWorkPosting.ReferencePostingType; join restricts this to ITMPOST. | Never missing in this extract. |
| REFERENCEPOSTINGID | identifier | 0 | 0.00% | 194,271 | 1954; 2008; 2007 | 6565 (2), 6566 (2), 6567 (2) | PROCMGTWorkPosting.ReferencePostingID. | Never missing in this extract. |
| POSTINGSTATUS | categorical/text | 0 | 0.00% | 2 | Approved; Closed | Approved (194,297), Closed (12) | PROCMGTWorkPosting.Status; query filters to non-null. | Never missing in this extract. |
| POSTINGDATE | date/time | 0 | 0.00% | 20,468 | 2017-03-02 07:00:00.000; 2017-03-15 06:00:00.000; 2017-03-17 06:00:00.000 | 2025-06-30 06:00:00.000 (558), 2020-06-30 06:00:00.000 (445), 2021-06-30 06:00:00.000 (419) | PROCMGTWorkPosting.PostingDate. | Never missing in this extract. |
| PAYAPPID | identifier | 0 | 0.00% | 8,405 | 102; 113; 152 | 3268 (636), 7171 (560), 11216 (495) | PROCMGTPayEstimates.PEID through PEDetails. | Never missing in this extract. |
| PAYAPPNUMBER | categorical/text | 0 | 0.00% | 50 | 4; 1; 2 | 2 (30,075), 3 (27,657), 1 (23,627) | PROCMGTPayEstimates.PayEstimateNumber. | Never missing in this extract. |
| PAYAPPSEQUENCE | quantity/number | 0 | 0.00% | 50 | 4; 1; 2 | 2 (30,075), 3 (27,657), 1 (23,627) | PROCMGTPayEstimates.PENum. | Never missing in this extract. |
| PAYAPPSTATUS | categorical/text | 0 | 0.00% | 3 | Approved; Submitted; Draft | Approved (193,913), Submitted (227), Draft (169) | PROCMGTPayEstimates.Status; query filters to non-null. | Never missing in this extract. |
| FROMDATE | date/time | 0 | 0.00% | 1,979 | 2017-03-01 07:00:00.000; 2017-03-06 07:00:00.000; 2017-04-01 06:00:00.000 | 2018-07-01 06:00:00.000 (2,619), 2019-07-01 06:00:00.000 (2,506), 2025-07-01 06:00:00.000 (2,340) | PROCMGTPayEstimates.FromDate. | Never missing in this extract. |
| TODATE | date/time | 0 | 0.00% | 1,683 | 2017-03-15 06:00:00.000; 2017-03-24 06:00:00.000; 2017-04-29 06:00:00.000 | 2025-06-30 06:00:00.000 (2,769), 2018-06-30 06:00:00.000 (2,680), 2019-06-30 06:00:00.000 (2,254) | PROCMGTPayEstimates.ToDate. | Never missing in this extract. |
| PEDETAILSID | identifier | 0 | 0.00% | 194,309 | 3181; 3182; 3183 | 3181 (1), 3182 (1), 3183 (1) | PROCMGTPEDetails.PEDetailsID. | Never missing in this extract. |
| BILLINGRATE | money/rate | 0 | 0.00% | 16,532 | 42.2500; 1.4500; 1.1500 | 1.0000 (2,128), 10.0000 (1,863), 100.0000 (1,111) | PROCMGTPEDetails.BillingRate. | Never missing in this extract. |
| QTYBILLEDTHISPAYAPP | quantity/number | 0 | 0.00% | 46,688 | 2903.0000; 9969.0000; 6282.9000 | 1.0000 (32,596), 2.0000 (9,333), 0.2500 (5,890) | PROCMGTPEDetails.Quantity. | Never missing in this extract. |
| BILLEDAMOUNT | money/rate | 0 | 0.00% | 91,053 | 122651.7500; 14455.0500; 7225.3400 | 0.0000 (3,402), 1500.0000 (539), 3000.0000 (523) | PROCMGTPEDetails.Amount. | Never missing in this extract. |
| TOTALWORKCOMPLETED | money/rate | 0 | 0.00% | 91,204 | 122651.75000000; 14455.05000000; 7225.33500000 | 0.00000000 (3,456), 1500.00000000 (539), 3000.00000000 (523) | Computed in SQL as CI.UnitPrice * PED.Quantity. | Never missing in this extract. |

## Missingness Ranked

| Field | Missing | Missing % | Unique non-missing |
| --- | --- | --- | --- |
| PHASEITEMSTARTDATE | 194,309 | 100.00% | 0 |
| PHASEITEMENDDATE | 194,309 | 100.00% | 0 |
| CONTRACTTYPE | 194,309 | 100.00% | 0 |
| CONTRACTITEMPHASENAME | 194,309 | 100.00% | 0 |
| LINKEDBUDGETITEMID | 194,309 | 100.00% | 0 |
| LINKEDBUDGETITEM | 194,309 | 100.00% | 0 |
| LINKEDBUDGETITEMMODULE | 194,309 | 100.00% | 0 |
| LINKEDBUDGETITEMDESCRIPTION | 194,309 | 100.00% | 0 |
| LINKEDBUDGETITEMCONTAINERPATH | 194,309 | 100.00% | 0 |
| LINKEDBUDGETITEMCONTAINERNAME | 194,309 | 100.00% | 0 |
| LINKEDBUDGETITEMPHASENAME | 194,309 | 100.00% | 0 |
| CONTRACTDESCRIPTION | 99,729 | 51.32% | 108 |
| PROJECTTYPE | 90,969 | 46.82% | 7 |
| PROJECTDESC | 163 | 0.08% | 154 |
| CONTRACTITEMCONTAINERPATH | 13 | 0.01% | 639 |
| CONTRACTITEMCONTAINERNAME | 13 | 0.01% | 520 |
| CONTRACTITEMUNIT | 13 | 0.01% | 37 |
| CUSTOMER | 0 | 0.00% | 1 |
| PROJECTNAME | 0 | 0.00% | 1,341 |
| PROJECTCODE | 0 | 0.00% | 1,346 |
| PROJECTSTATUS | 9 | 0.00% | 3 |
| CONTRACTID | 0 | 0.00% | 1,346 |
| CONTRACTNAME | 0 | 0.00% | 1,340 |
| CONTRACTSTARTDATE | 0 | 0.00% | 709 |
| CONTRACTENDDATE | 0 | 0.00% | 1,073 |
| PROJSTARTDATE | 0 | 0.00% | 341 |
| PROJENDDATE | 0 | 0.00% | 607 |
| WPPOSTINGDATE | 0 | 0.00% | 20,468 |
| ITEMID | 0 | 0.00% | 66,314 |
| STANDARDITEMNO | 0 | 0.00% | 9,081 |
| ITEMNAME | 0 | 0.00% | 17,332 |
| ITEMUNITPRICE | 0 | 0.00% | 16,532 |
| ITEMTOTALQUANTITY | 0 | 0.00% | 9,077 |
| CONTRACTITEMSYSTEMNAME | 0 | 0.00% | 1 |
| POSTINGID | 0 | 0.00% | 194,271 |
| POSTINGQTY | 0 | 0.00% | 46,698 |
| POSTINGUNITRATE | 0 | 0.00% | 16,532 |
| TOTALPOSTINGAMOUNT | 0 | 0.00% | 91,207 |
| REFERENCEPOSTINGTYPE | 0 | 0.00% | 1 |
| REFERENCEPOSTINGID | 0 | 0.00% | 194,271 |
| POSTINGSTATUS | 0 | 0.00% | 2 |
| POSTINGDATE | 0 | 0.00% | 20,468 |
| PAYAPPID | 0 | 0.00% | 8,405 |
| PAYAPPNUMBER | 0 | 0.00% | 50 |
| PAYAPPSEQUENCE | 0 | 0.00% | 50 |
| PAYAPPSTATUS | 0 | 0.00% | 3 |
| FROMDATE | 0 | 0.00% | 1,979 |
| TODATE | 0 | 0.00% | 1,683 |
| PEDETAILSID | 0 | 0.00% | 194,309 |
| BILLINGRATE | 0 | 0.00% | 16,532 |
| QTYBILLEDTHISPAYAPP | 0 | 0.00% | 46,688 |
| BILLEDAMOUNT | 0 | 0.00% | 91,053 |
| TOTALWORKCOMPLETED | 0 | 0.00% | 91,204 |

## Date Fields and Ranges

| Field | Parsed | Missing | Parse failures | Min | Q1-ish | Median-ish | Q3-ish | Max | Interpretation |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| CONTRACTSTARTDATE | 194,309 | 0 | 0 | 1900-01-01 07:00:00 | 2018-07-09 06:00:00 | 2020-05-19 06:00:00 | 2023-06-07 06:00:00 | 2026-04-15 06:00:00 | CONTMGTMaster.StartDt. |
| CONTRACTENDDATE | 194,309 | 0 | 0 | 1900-01-10 07:00:00 | 2019-01-31 00:00:00 | 2020-12-19 00:00:00 | 2023-11-01 00:00:00 | 2026-05-21 00:00:00 | CONTMGTMaster.CLOSUREDT; query filters ContractEndDate < CURRENT_DATE. |
| PROJSTARTDATE | 194,309 | 0 | 0 | 1900-01-01 07:00:00 | 1900-01-01 07:00:00 | 1900-01-01 07:00:00 | 2021-08-10 06:00:00 | 2026-05-25 06:00:00 | PROJECTPROJECTMAIN.STARTDATE. |
| PROJENDDATE | 194,309 | 0 | 0 | 1899-12-31 07:00:00 | 1900-07-31 06:00:00 | 2021-05-31 06:00:00 | 2025-06-12 06:00:00 | 2100-01-01 07:00:00 | PROJECTPROJECTMAIN.ENDDATE. |
| PHASEITEMSTARTDATE | 0 | 194,309 | 0 |  |  |  |  |  | PROJECTProjectPhaseItem.STARTDATE through contract-item container hierarchy. |
| PHASEITEMENDDATE | 0 | 194,309 | 0 |  |  |  |  |  | PROJECTProjectPhaseItem.ENDDATE through contract-item container hierarchy. |
| WPPOSTINGDATE | 194,309 | 0 | 0 | 1981-03-06 07:00:00 | 2019-01-17 16:17:00 | 2021-02-18 07:00:00 | 2023-10-14 06:00:00 | 2026-05-08 06:00:00 | PROCMGTWorkPosting.POSTINGDATE, selected separately from POSTINGDATE. |
| POSTINGDATE | 194,309 | 0 | 0 | 1981-03-06 07:00:00 | 2019-01-17 16:17:00 | 2021-02-18 07:00:00 | 2023-10-14 06:00:00 | 2026-05-08 06:00:00 | PROCMGTWorkPosting.PostingDate. |
| FROMDATE | 194,309 | 0 | 0 | 1899-05-20 06:00:00 | 2018-12-19 07:00:00 | 2021-01-13 07:00:00 | 2023-09-24 06:00:00 | 2026-04-19 06:00:00 | PROCMGTPayEstimates.FromDate. |
| TODATE | 194,309 | 0 | 0 | 2016-07-30 06:00:00 | 2019-02-16 07:00:00 | 2021-03-13 07:00:00 | 2023-10-31 06:00:00 | 2026-05-11 06:00:00 | PROCMGTPayEstimates.ToDate. |

## Date Relationship Checks

These checks help separate business semantics from data quality issues. `WPPOSTINGDATE` and `POSTINGDATE` are the same selected source column in the SQL, so they should match exactly. `FROMDATE` and `TODATE` describe the pay estimate period, which may not perfectly contain the posting date depending on operational practice.

| Check | OK rows | Exception rows | Exception % |
| --- | --- | --- | --- |
| contract_start_lte_end | 192,899 | 1,410 | 0.73% |
| payapp_from_lte_to | 194,309 | 0 | 0.00% |
| posting_inside_contract_window | 114,556 | 79,753 | 41.04% |
| posting_inside_payapp_window | 178,595 | 15,714 | 8.09% |
| posting_inside_project_window | 68,309 | 126,000 | 64.85% |
| project_start_lte_end | 183,407 | 10,902 | 5.61% |
| wpposting_equals_posting | 194,309 | 0 | 0.00% |

### Durations and Date Differences

| Difference | Rows | Min days | Median days | Max days | Mean days |
| --- | --- | --- | --- | --- | --- |
| CONTRACTENDDATE_minus_CONTRACTSTARTDATE_days | 194,309 | -42,843.25 | 136.71 | 1,252.75 | 167.49 |
| PROJENDDATE_minus_PROJSTARTDATE_days | 194,309 | -44,001.96 | 191.96 | 73,049.00 | 15,782.61 |
| TODATE_minus_FROMDATE_days | 194,309 | 0.00 | 31.00 | 43,141.00 | 51.97 |
| WPPOSTINGDATE_minus_POSTINGDATE_days | 194,309 | 0.00 | 0.00 | 0.00 | 0.00 |

## Numeric Fields and Ranges

| Field | Parsed | Missing | Parse failures | Min | P05 | P25 | Median | P75 | P95 | Max | Sum |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| CONTRACTID | 194,309 | 0 | 0 | 5.0000 | 196.0000 | 590.0000 | 902.0000 | 1,327.0000 | 1,673.0000 | 1,901.0000 | 181,830,935.00 |
| ITEMID | 194,309 | 0 | 0 | 1,052.0000 | 19,886.0000 | 35,790.0000 | 60,991.0000 | 88,464.0000 | 107,444.0000 | 124,951.0000 | 12,025,616,483.00 |
| ITEMUNITPRICE | 194,309 | 0 | 0 | -5,655.0000 | 1.4000 | 22.2500 | 120.0000 | 1,592.7500 | 100,000.0000 | 347,727,485.3300 | 41,269,836,562.00 |
| ITEMTOTALQUANTITY | 194,309 | 0 | 0 | 0.0000 | 1.0000 | 4.0000 | 104.0000 | 3,021.0000 | 91,528.8000 | 465,965,029.0000 | 64,535,466,415.84 |
| LINKEDBUDGETITEMID | 0 | 194,309 | 0 |  |  |  |  |  |  |  |  |
| POSTINGID | 194,309 | 0 | 0 | 9.0000 | 11,130.4000 | 53,659.0000 | 107,349.0000 | 160,531.0000 | 206,657.6000 | 225,197.0000 | 20,944,164,153.00 |
| POSTINGQTY | 194,309 | 0 | 0 | -2,980,400.0000 | 0.0900 | 1.0000 | 12.0000 | 261.6300 | 6,417.1620 | 29,206,013.0000 | 2,034,753,580.54 |
| POSTINGUNITRATE | 194,309 | 0 | 0 | -5,655.0000 | 1.4000 | 22.2500 | 120.0000 | 1,592.7500 | 100,000.0000 | 347,727,485.3300 | 41,269,836,562.00 |
| TOTALPOSTINGAMOUNT | 194,309 | 0 | 0 | -12,764,070.0000 | 73.5000 | 840.0000 | 3,744.4000 | 15,637.5000 | 117,305.5960 | 36,000,000.0000 | 6,957,314,366.91 |
| REFERENCEPOSTINGID | 194,309 | 0 | 0 | 22.0000 | 11,817.4000 | 55,204.0000 | 109,205.0000 | 164,243.0000 | 210,873.6000 | 231,884.0000 | 21,426,549,083.00 |
| PAYAPPID | 194,309 | 0 | 0 | 24.0000 | 540.0000 | 2,540.0000 | 5,087.0000 | 8,094.0000 | 10,717.0000 | 11,546.0000 | 1,040,653,979.00 |
| PAYAPPSEQUENCE | 194,309 | 0 | 0 | 1.0000 | 1.0000 | 2.0000 | 4.0000 | 8.0000 | 20.0000 | 52.0000 | 1,251,003.00 |
| PEDETAILSID | 194,309 | 0 | 0 | 25.0000 | 28,979.4000 | 167,413.0000 | 353,116.0000 | 517,775.0000 | 668,011.6000 | 739,702.0000 | 67,314,018,642.00 |
| BILLINGRATE | 194,309 | 0 | 0 | -5,655.0000 | 1.4000 | 22.2500 | 120.0000 | 1,592.7500 | 100,000.0000 | 347,727,485.3300 | 41,269,836,562.00 |
| QTYBILLEDTHISPAYAPP | 194,309 | 0 | 0 | -2,980,400.0000 | 0.0900 | 1.0000 | 12.0000 | 261.6300 | 6,417.1620 | 29,206,013.0000 | 2,034,753,580.54 |
| BILLEDAMOUNT | 194,309 | 0 | 0 | -12,764,070.0000 | 74.0400 | 840.0000 | 3,750.0000 | 15,660.0000 | 117,527.7280 | 36,000,000.0000 | 7,010,714,161.25 |
| TOTALWORKCOMPLETED | 194,309 | 0 | 0 | -12,764,070.0000 | 73.5000 | 840.0000 | 3,744.4000 | 15,637.5000 | 117,305.5960 | 36,000,000.0000 | 6,957,313,549.86 |

## Amount Consistency Checks

The SQL computes `TOTALPOSTINGAMOUNT` as `POSTINGQTY * POSTINGUNITRATE` and `TOTALWORKCOMPLETED` as `ITEMUNITPRICE * QTYBILLEDTHISPAYAPP`. `BILLEDAMOUNT` comes from pay estimate details and is expected to line up with `BILLINGRATE * QTYBILLEDTHISPAYAPP` when rates are populated consistently.

| Check | Matches | Mismatches | Mismatch % |
| --- | --- | --- | --- |
| posting_amount | 194,309 | 0 | 0.00% |
| billed_amount | 193,618 | 691 | 0.36% |
| total_work_completed | 194,309 | 0 | 0.00% |

## Categorical and Text Fields

| Field | Missing | Missing % | Unique | Top values |
| --- | --- | --- | --- | --- |
| CUSTOMER | 0 | 0.00% | 1 | UDOT (194,309, 100.00%) |
| PROJECTNAME | 0 | 0.00% | 1,341 | I-15; SR-232 to I-84 (4,014, 2.07%); SR-30; SR-23 to SR-252 (3,497, 1.80%); SR-108; 300 North to 1800 North (2,207, 1.14%); SR-18; St. George Blvd. to Sunset Blvd. (1,937, 1.00%); SR-68; Bangerter Hwy to 12600 S (1,591, 0.82%); Enhanced Freeway Striping (1,517, 0.78%); US-89 (500 W); 500 S to Bulldog Blvd in Provo (1,463, 0.75%); SR-99; Fillmore Main Street (1,445, 0.74%) |
| PROJECTCODE | 0 | 0.00% | 1,346 | F-I15-7(328)332 (4,014, 2.07%); F-0030(69)102 (3,497, 1.80%); S-0108(36)6 (2,207, 1.14%); S-0018(49)2 (1,937, 1.00%); S-0068(95)41 (1,591, 0.82%); F-ST99(799) (1,517, 0.78%); F-0089(446)335 (1,463, 0.75%); F-0099(7)0 (1,445, 0.74%) |
| PROJECTDESC | 163 | 0.08% | 154 | Preservation High Volume (26,265, 13.52%); Rehabilitation High Volume (22,016, 11.33%); Widen Existing Facility (17,167, 8.83%); Choke Point (12,548, 6.46%); Reconstruct & Widening (6,290, 3.24%); New Capacity (6,207, 3.19%); Intersection Modification (5,874, 3.02%); Passing Lane (4,987, 2.57%) |
| PROJECTTYPE | 90,969 | 46.82% | 7 | Design Bid Build (95,410, 49.10%); CMGC (5,164, 2.66%); Design Build (2,036, 1.05%); Progressive Design Build (355, 0.18%); Structural Pool (199, 0.10%); Other (96, 0.05%); Procurement (80, 0.04%) |
| PROJECTSTATUS | 9 | 0.00% | 3 | Construction (192,688, 99.17%); Complete (1,598, 0.82%); Notice To Proceed (NTP) (14, 0.01%) |
| CONTRACTNAME | 0 | 0.00% | 1,340 | I-15; SR-232 to I-84 (4,014, 2.07%); SR-30; SR-23 to SR-252 (3,497, 1.80%); SR-108; 300 North to 1800 North (2,207, 1.14%); SR-18; St. George Blvd. to Sunset Blvd. (1,937, 1.00%); SR-68; Bangerter Hwy to 12600 S (1,591, 0.82%); Enhanced Freeway Striping (1,517, 0.78%); US-89 (500 W); 500 S to Bulldog Blvd in Provo (1,463, 0.75%); SR-99; Fillmore Main Street (1,445, 0.74%) |
| CONTRACTDESCRIPTION | 99,729 | 51.32% | 108 | Preservation High Volume (17,653, 9.09%); Widen Existing Facility (9,733, 5.01%); Rehabilitation High Volume (8,366, 4.31%); Intersection Improvements (3,491, 1.80%); Choke Point (3,338, 1.72%); Other-Roadway Project (2,738, 1.41%); Rehabilitation Low Volume (2,519, 1.30%); TIF - Transportation Investment Fund (2,517, 1.30%) |
| CONTRACTTYPE | 194,309 | 100.00% | 0 |  |
| STANDARDITEMNO | 0 | 0.00% | 9,081 | 015017010 (3,094, 1.59%); 01554700* (2,273, 1.17%); 028917320 (2,147, 1.10%); 015547005 (2,136, 1.10%); 017217010 (2,108, 1.08%); 028917270 (2,107, 1.08%); 015407010 (1,639, 0.84%); 02741705* (1,598, 0.82%) |
| ITEMNAME | 0 | 0.00% | 17,332 | Traffic Control (5,425, 2.79%); HMA - 1/2 inch (4,411, 2.27%); Mobilization (3,759, 1.93%); Public Information Services (3,538, 1.82%); Survey (3,433, 1.77%); Pavement Marking Paint (2,646, 1.36%); Remove Sign Less Than 20 Square Feet (2,539, 1.31%); Slipbase Sign Base (B3) (2,494, 1.28%) |
| CONTRACTITEMCONTAINERPATH | 13 | 0.01% | 639 | 10 - ROADWAY (127,649, 65.69%); 40 - SIGNING (28,258, 14.54%); 20 - STRUCTURES (4,638, 2.39%); 50 - SIGNALS (3,909, 2.01%); 70 - ATMS (3,880, 2.00%); 30 - LANDSCAPING (3,481, 1.79%); 80 - NON-PARTICIPATING (1,696, 0.87%); 10 - ROADWAY -    (999, 0.51%) |
| CONTRACTITEMCONTAINERNAME | 13 | 0.01% | 520 | 10 - ROADWAY (128,106, 65.93%); 40 - SIGNING (28,286, 14.56%); 20 - STRUCTURES (4,673, 2.40%); 50 - SIGNALS (3,922, 2.02%); 70 - ATMS (3,897, 2.01%); 30 - LANDSCAPING (3,483, 1.79%); 80 - NON-PARTICIPATING (1,696, 0.87%); 10 - ROADWAY -    (999, 0.51%) |
| CONTRACTITEMSYSTEMNAME | 0 | 0.00% | 1 | IS System (194,309, 100.00%) |
| CONTRACTITEMUNIT | 13 | 0.01% | 37 | Each (62,796, 32.32%); ft (24,794, 12.76%); Lump (17,553, 9.03%); Ton (15,480, 7.97%); foot (14,579, 7.50%); sq ft (12,547, 6.46%); lump sum (10,580, 5.44%); cu yd (10,458, 5.38%) |
| CONTRACTITEMPHASENAME | 194,309 | 100.00% | 0 |  |
| LINKEDBUDGETITEM | 194,309 | 100.00% | 0 |  |
| LINKEDBUDGETITEMMODULE | 194,309 | 100.00% | 0 |  |
| LINKEDBUDGETITEMDESCRIPTION | 194,309 | 100.00% | 0 |  |
| LINKEDBUDGETITEMCONTAINERPATH | 194,309 | 100.00% | 0 |  |
| LINKEDBUDGETITEMCONTAINERNAME | 194,309 | 100.00% | 0 |  |
| LINKEDBUDGETITEMPHASENAME | 194,309 | 100.00% | 0 |  |
| REFERENCEPOSTINGTYPE | 0 | 0.00% | 1 | ITMPOST (194,309, 100.00%) |
| POSTINGSTATUS | 0 | 0.00% | 2 | Approved (194,297, 99.99%); Closed (12, 0.01%) |
| PAYAPPNUMBER | 0 | 0.00% | 50 | 2 (30,075, 15.48%); 3 (27,657, 14.23%); 1 (23,627, 12.16%); 4 (20,889, 10.75%); 5 (15,666, 8.06%); 6 (13,223, 6.81%); 7 (9,895, 5.09%); 8 (8,023, 4.13%) |
| PAYAPPSTATUS | 0 | 0.00% | 3 | Approved (193,913, 99.80%); Submitted (227, 0.12%); Draft (169, 0.09%) |

## Grain and Entity Counts

| Metric | Value |
| --- | --- |
| Rows | 194,309 |
| Unique projects | 1,346 |
| Unique contracts | 1,346 |
| Unique contract items | 66,314 |
| Unique postings | 194,271 |
| Unique pay apps | 8,405 |
| Unique pay estimate detail rows | 194,309 |
| Unique item/pay-app pairs | 121,794 |

### Contracts With Most Distinct Items

| CONTRACTID | Distinct ITEMID count |
| --- | --- |
| 747 | 573 |
| 1416 | 384 |
| 428 | 375 |
| 1326 | 350 |
| 817 | 343 |
| 477 | 309 |
| 1562 | 309 |
| 893 | 306 |
| 898 | 299 |
| 669 | 293 |
| 143 | 286 |
| 761 | 271 |
| 803 | 271 |
| 1352 | 268 |
| 592 | 257 |
| 1008 | 257 |
| 703 | 257 |
| 1472 | 254 |
| 767 | 253 |
| 396 | 250 |

### Number of Pay Apps per Contract Item

| Distinct pay apps on item | Item count | Share of items |
| --- | --- | --- |
| 1 | 40,727 | 61.42% |
| 2 | 13,443 | 20.27% |
| 3 | 5,667 | 8.55% |
| 4 | 2,843 | 4.29% |
| 5 | 1,418 | 2.14% |
| 6 | 755 | 1.14% |
| 7 | 478 | 0.72% |
| 8 | 307 | 0.46% |
| 9 | 182 | 0.27% |
| 10 | 131 | 0.20% |
| 11 | 93 | 0.14% |
| 12 | 55 | 0.08% |
| 13 | 51 | 0.08% |
| 14 | 44 | 0.07% |
| 15 | 29 | 0.04% |
| 16 | 10 | 0.02% |
| 17 | 11 | 0.02% |
| 18 | 12 | 0.02% |
| 19 | 8 | 0.01% |
| 20 | 9 | 0.01% |
| 21 | 6 | 0.01% |
| 22 | 3 | 0.00% |
| 23 | 4 | 0.01% |
| 24 | 4 | 0.01% |
| 25 | 1 | 0.00% |
| 26 | 5 | 0.01% |
| 27 | 3 | 0.00% |
| 28 | 1 | 0.00% |
| 29 | 1 | 0.00% |
| 31 | 1 | 0.00% |

## High-Value and Standard Item Patterns

These tables are useful for deciding which item categories or large-dollar lines deserve follow-up modeling or manual review.

### Highest Total Billed Items

| ITEMID | Total billed amount | Rows | Distinct pay apps |
| --- | --- | --- | --- |
| 68698 | 465,965,028.98 | 42 | 42 |
| 54059 | 347,727,485.33 | 46 | 46 |
| 42512 | 219,090,000.00 | 36 | 35 |
| 93523 | 215,887,043.60 | 29 | 29 |
| 29914 | 182,944,921.96 | 38 | 38 |
| 96813 | 166,099,222.07 | 29 | 26 |
| 57754 | 141,235,000.00 | 35 | 32 |
| 94749 | 105,854,518.12 | 26 | 26 |
| 32539 | 96,342,969.86 | 73 | 37 |
| 44374 | 95,993,710.00 | 31 | 31 |
| 71520 | 91,456,604.00 | 58 | 34 |
| 34441 | 40,300,000.00 | 3 | 1 |
| 29913 | 39,293,000.00 | 38 | 38 |
| 93319 | 36,000,000.00 | 1 | 1 |
| 96809 | 34,777,500.00 | 26 | 26 |
| 96810 | 33,500,000.00 | 15 | 15 |
| 19674 | 27,427,634.54 | 33 | 26 |
| 19677 | 27,146,725.59 | 25 | 24 |
| 19675 | 26,721,168.31 | 27 | 26 |
| 63936 | 25,056,115.16 | 28 | 27 |

### Standard Item Number Prefixes by Billed Amount

| STANDARDITEMNO prefix | Rows | Total billed amount |
| --- | --- | --- |
| 00122 | 930 | 2,176,729,937.52 |
| 02741 | 6,812 | 613,339,872.34 |
| 02744 | 2,480 | 372,667,642.41 |
| 00000 | 735 | 350,801,903.47 |
| 01501 | 3,682 | 300,040,456.95 |
| 01554 | 6,410 | 239,954,628.37 |
| 02056 | 3,531 | 136,197,497.98 |
| 02316 | 2,794 | 122,126,481.23 |
| 02721 | 3,817 | 112,434,945.90 |
| 02776 | 9,898 | 93,476,546.91 |
| 03310 | 1,271 | 92,159,362.38 |
| 02610 | 6,521 | 90,849,625.80 |
| 02735 | 1,540 | 88,942,553.15 |
| 02844 | 2,208 | 78,530,645.57 |
| 02221 | 16,577 | 74,068,785.56 |
| 02785 | 867 | 69,364,144.50 |
| 02752 | 637 | 69,343,660.94 |
| 02961 | 3,197 | 62,299,767.74 |
| 02633 | 5,531 | 53,366,957.86 |
| 13553 | 2,102 | 52,115,007.17 |
| 02740 | 406 | 48,674,108.48 |
| 02787 | 376 | 46,521,035.05 |
| 02768 | 5,042 | 40,238,552.36 |
| 05120 | 249 | 40,165,890.75 |
| 02891 | 23,035 | 39,953,644.70 |

## Practical Interpretation of Dates

- `CONTRACTSTARTDATE` and `CONTRACTENDDATE` come from the contract master. Because the SQL filters closed contracts with `ContractEndDate < CURRENT_DATE`, this dataset intentionally emphasizes completed contracts.
- `PROJSTARTDATE` and `PROJENDDATE` come from the project master. They are broader project dates and can differ from contract dates.
- `PHASEITEMSTARTDATE` and `PHASEITEMENDDATE` are inherited through the contract-item container hierarchy. Missing values usually mean the item container is not tied to a phase item with dates, not that the contract item itself is missing.
- `WPPOSTINGDATE` and `POSTINGDATE` are both selected from `PROCMGTWorkPosting.POSTINGDATE`; the duplicate fields are redundant in this export.
- `FROMDATE` and `TODATE` are pay estimate period boundaries. They are repeated across many item/detail rows sharing the same `PAYAPPID`.


## Follow-Up Analysis Ideas

- Aggregate to one row per `ITEMID` for production curves, because the current row grain repeats item attributes across pay app/detail rows.
- Aggregate to one row per `PAYAPPID` to study pay estimate cycles and period lengths.
- Investigate linked budget missingness separately. Budget fields are left-joined and are structurally optional; missingness here does not necessarily imply a broken payment record.
- Review rows where amount checks mismatch or date checks fail before using the data for forecasting or training.